# Analyse statistique des variantes SegFormer

Cette analyse complète la comparaison en validation croisée entre
SegFormer-B0, SegFormer-B1 et SegFormer-B2.

Les trois variantes utilisent les mêmes cinq partitions stratifiées
des 376 images de développement. Chaque modèle issu d'un fold est
ensuite évalué sur le même ensemble de test indépendant de 94 images.

L'objectif est de déterminer si les faibles écarts de mIoU observés
entre B0, B1 et B2 dépassent la variabilité liée aux différents
entraînements et partitions de validation.

VGG16-FCN8 n'est pas inclus dans le test statistique apparié, car ses
partitions de validation croisée ne correspondent pas aux partitions
stratifiées utilisées pour les variantes SegFormer.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from scipy.stats import (
    t,
    friedmanchisquare
)

PROJECT_ROOT = Path(
    r"C:\M2_USPN\Projet_Stage_M2\implémentations"
    r"\Posidonia_Oceanica_Detection_Project"
)

SEGFORMER_ROOT = PROJECT_ROOT / "segformer"

KFOLD_DIRS = {
    "B0": SEGFORMER_ROOT / "segformer_b0" / "results" / "kfold",
    "B1": SEGFORMER_ROOT / "segformer_b1" / "results" / "kfold",
    "B2": SEGFORMER_ROOT / "segformer_b2" / "results" / "kfold",
}

for model, path in KFOLD_DIRS.items():
    print(model, ":", path)
    print("  Existe :", path.exists())

B0 : C:\M2_USPN\Projet_Stage_M2\implémentations\Posidonia_Oceanica_Detection_Project\segformer\segformer_b0\results\kfold
  Existe : True
B1 : C:\M2_USPN\Projet_Stage_M2\implémentations\Posidonia_Oceanica_Detection_Project\segformer\segformer_b1\results\kfold
  Existe : True
B2 : C:\M2_USPN\Projet_Stage_M2\implémentations\Posidonia_Oceanica_Detection_Project\segformer\segformer_b2\results\kfold
  Existe : True


In [4]:
def load_kfold_metadata(model_name, directory):

    rows = []

    prefix_map = {
        "B0": "segformerb0",
        "B1": "segformerb1",
        "B2": "segformerb2",
    }

    prefix = prefix_map[model_name]

    for fold in range(5):

        path = directory / (
            f"{prefix}_kfold_fold{fold}_ce_dice_metadata.json"
        )

        if not path.exists():
            print(
                f"[MANQUANT] {model_name} fold {fold} : "
                f"{path.name}"
            )
            continue

        print(
            f"[OK] {model_name} fold {fold} : "
            f"{path.name}"
        )

        with open(path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        rows.append({
            "model": model_name,
            "fold": fold,

            "val_threshold": float(
                meta["val_threshold"]
            ),

            "test_threshold": float(
                meta["test_threshold"]
            ),

            "val_miou": float(
                meta["val_metrics"]["miou"]
            ),

            "test_miou": float(
                meta["test_metrics"]["miou"]
            ),

            "test_dice": float(
                meta["test_metrics"]["dice"]
            ),

            "test_accuracy": float(
                meta["test_metrics"]["accuracy"]
            ),
        })

    return rows

In [5]:
rows = []

for model_name, directory in KFOLD_DIRS.items():

    rows.extend(
        load_kfold_metadata(
            model_name,
            directory
        )
    )

df = pd.DataFrame(rows)

print(
    f"\nNombre de résultats chargés : {len(df)}"
)

display(df)

[OK] B0 fold 0 : segformerb0_kfold_fold0_ce_dice_metadata.json
[OK] B0 fold 1 : segformerb0_kfold_fold1_ce_dice_metadata.json
[OK] B0 fold 2 : segformerb0_kfold_fold2_ce_dice_metadata.json
[OK] B0 fold 3 : segformerb0_kfold_fold3_ce_dice_metadata.json
[OK] B0 fold 4 : segformerb0_kfold_fold4_ce_dice_metadata.json
[OK] B1 fold 0 : segformerb1_kfold_fold0_ce_dice_metadata.json
[OK] B1 fold 1 : segformerb1_kfold_fold1_ce_dice_metadata.json
[OK] B1 fold 2 : segformerb1_kfold_fold2_ce_dice_metadata.json
[OK] B1 fold 3 : segformerb1_kfold_fold3_ce_dice_metadata.json
[OK] B1 fold 4 : segformerb1_kfold_fold4_ce_dice_metadata.json
[OK] B2 fold 0 : segformerb2_kfold_fold0_ce_dice_metadata.json
[OK] B2 fold 1 : segformerb2_kfold_fold1_ce_dice_metadata.json
[OK] B2 fold 2 : segformerb2_kfold_fold2_ce_dice_metadata.json
[OK] B2 fold 3 : segformerb2_kfold_fold3_ce_dice_metadata.json
[OK] B2 fold 4 : segformerb2_kfold_fold4_ce_dice_metadata.json

Nombre de résultats chargés : 15


,model,fold,val_threshold,test_threshold,val_miou,test_miou,test_dice,test_accuracy
0,B0,0,0.5,0.5,0.939574,0.939601,0.965741,0.969161
1,B0,1,0.5,0.5,0.920035,0.933437,0.962108,0.965908
2,B0,2,0.4,0.4,0.931414,0.938940,0.965434,0.968798
3,B0,3,0.4,0.4,0.929710,0.934476,0.962895,0.966425
4,B0,4,0.4,0.4,0.935979,0.940092,0.966069,0.969412
5,B1,0,0.4,0.5,0.937874,0.940435,0.966177,0.969610
6,B1,1,0.4,0.3,0.924260,0.938732,0.965350,0.968681
7,B1,2,0.5,0.4,0.931548,0.938152,0.965025,0.968373
8,B1,3,0.4,0.5,0.934956,0.940454,0.966201,0.969618
9,B1,4,0.4,0.4,0.936667,0.940782,0.966497,0.969770


In [6]:
pivot_val = df.pivot(
    index="fold",
    columns="model",
    values="val_miou"
)

print("mIoU de validation par fold (%)")
display((pivot_val * 100).round(4))

mIoU de validation par fold (%)


model,B0,B1,B2
fold,,,
0,93.9574,93.7874,93.9434
1,92.0035,92.4260,92.4466
2,93.1414,93.1548,93.4553
3,92.9710,93.4956,93.3773
4,93.5979,93.6667,93.6414


Les statistiques descriptives

In [7]:
from scipy.stats import t
import numpy as np
import pandas as pd

def descriptive_statistics(values):
    values = np.asarray(values, dtype=float)

    n = len(values)
    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_crit = t.ppf(0.975, df=n - 1)
    margin = t_crit * se

    return {
        "mean": mean,
        "sd": sd,
        "median": np.median(values),
        "ci_low": mean - margin,
        "ci_high": mean + margin,
    }


summary = []

for model in ["B0", "B1", "B2"]:
    stats = descriptive_statistics(
        pivot_val[model].values
    )

    summary.append({
        "Modèle": f"SegFormer-{model}",
        "mIoU moyen (%)": stats["mean"] * 100,
        "Écart-type (%)": stats["sd"] * 100,
        "Médiane (%)": stats["median"] * 100,
        "IC95 bas (%)": stats["ci_low"] * 100,
        "IC95 haut (%)": stats["ci_high"] * 100,
    })

df_stats = pd.DataFrame(summary)

display(df_stats.round(3))

,Modèle,mIoU moyen (%),Écart-type (%),Médiane (%),IC95 bas (%),IC95 haut (%)
0,SegFormer-B0,93.134,0.741,93.141,92.214,94.055
1,SegFormer-B1,93.306,0.547,93.496,92.627,93.985
2,SegFormer-B2,93.373,0.562,93.455,92.675,94.070


Le test de Friedman

In [8]:
from scipy.stats import friedmanchisquare

friedman_stat, friedman_p = friedmanchisquare(
    pivot_val["B0"].values,
    pivot_val["B1"].values,
    pivot_val["B2"].values
)

print(
    f"Statistique de Friedman = "
    f"{friedman_stat:.4f}"
)

print(
    f"p-value = "
    f"{friedman_p:.6f}"
)

alpha = 0.05

if friedman_p < alpha:
    print(
        "Différence globale statistiquement significative."
    )
else:
    print(
        "Aucune différence globale statistiquement "
        "significative au seuil de 5 %."
    )

Statistique de Friedman = 2.8000
p-value = 0.246597
Aucune différence globale statistiquement significative au seuil de 5 %.


Kendall W

In [9]:
n = pivot_val.shape[0]
k = pivot_val.shape[1]

kendall_w = (
    friedman_stat
    / (n * (k - 1))
)

print(
    f"Kendall W = {kendall_w:.4f}"
)

Kendall W = 0.2800
